In [37]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import optuna # very nice library for hyperparam tuning
from sklearn.metrics import f1_score
import copy

In [26]:
train_folder = Path("train")
test_folder = Path("test")

df = pd.read_csv("train.csv")
image_id_col, class_col = df.columns[0], df.columns[1]

# check what class labels are
print(df[class_col].unique())

# check for imbalance
print(df[class_col].value_counts())


[6 2 5 3 0 1 7 4]
label
6    2663
1    2494
3    2316
7    1878
2    1241
5    1136
0     974
4     971
Name: count, dtype: int64


classes are 0-7, theres slight imbalance in the classes where some occur way less than others

In [27]:
# check some properties of the images, will only use 1 image and assume the rest are the same hopefully
test_image = Image.open("C:\\Users\\lina ahmed\\kaggle stuff\\train\\0a0e80ce71.png")
print(test_image.mode)
print(test_image.size)

RGB
(28, 28)


In [28]:
# custom dataset since our classes are in the csv not the image folder, so regular dataset wont work
class CellDataset(Dataset):
    def __init__(self, frame, root, transform=None):
        self.files = frame[image_id_col].tolist()
        self.labels = frame["label"].tolist() if "label" in frame else None  # works for labeled and unlabeled data
        self.root, self.transform = Path(root), transform

    def __len__(self):
        return len(self.files)  # returns length for batch norm later

    def __getitem__(self, i):
        img = Image.open(self.root / self.files[i]).convert("RGB")  # opens image path then gets specific image, convert rgb to make sure theyre all rgb
        if self.transform:
            img = self.transform(img)
        return (img, self.labels[i]) if self.labels is not None else img # returns no label if it doesnt exist for the test

In [29]:
# get val dataset too
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42)

# get mean and std of train
tmp = CellDataset(train_df, train_folder, T.ToTensor()) # T.totensor turns the images into torch tensors, 
                                                        # tmp[i] returns (image, label) with the dim in the correct order for cnn

images = torch.stack([tmp[i][0] for i in range(len(tmp))])   # N x 3 x 28 x 28 for the cnn

mean, std = images.mean(dim=(0, 2, 3)).tolist(), images.std(dim=(0, 2, 3)).tolist() # dims is 0 for images, 2 for height, 3 for width, but keeps 1 (channels) seperate
# so this will return a mean and std for each colour channel

In [30]:
# transform

train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomChoice([T.RandomRotation((a, a)) for a in (0, 90, 180, 270)]),
    T.RandomCrop(28, padding=2, padding_mode="reflect"),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

In [31]:
train_ds = CellDataset(train_df, train_folder, train_tf)
val_ds   = CellDataset(val_df, train_folder, eval_tf)

train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)

counts = np.bincount(train_df["label"], minlength=8)
class_weights = torch.tensor(counts.sum() / (8 * counts), dtype=torch.float)  # for class imbalance later when training

In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def block(cin, cout, act): # helper class so i dont repeat everything
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), act(),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), act(),
        nn.MaxPool2d(2))

class CNN(nn.Module):
    def __init__(self, n_classes=8, act=nn.ReLU, drop=0.3, width=32):
        super().__init__()
        w = width
        self.features = nn.Sequential(
            block(3, w, act),
            block(w, 2*w, act),
            block(2*w, 4*w, act))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(drop), nn.Linear(4*w, n_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
    def forward(self, x):
        return self.head(self.features(x))

model = CNN().to(device)

In [33]:
# create eval function to use in optuna later

@torch.no_grad()
def evaluate_with(model, loader):
    model.eval()
    total_loss, preds, labels = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += nn.functional.cross_entropy(out, y, reduction="sum").item()
        preds += out.argmax(1).cpu().tolist()
        labels += y.cpu().tolist()
    acc = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    return total_loss / len(labels), acc, f1_score(labels, preds, average="macro")

In [34]:
# optuna optimization

def objective(trial):
    # params to optimize for
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True) # learning rate
    wd = trial.suggest_float("wd", 1e-6, 1e-2, log=True) # weight decay
    drop = trial.suggest_float("drop", 0.0, 0.5) # dropout vals
    bs = trial.suggest_categorical("bs", [32, 64, 128]) # batch size
    act_name = trial.suggest_categorical("act", ["relu", "leaky", "gelu", "silu", "elu"]) # activation
    width = trial.suggest_categorical("width", [16, 32, 48])
    acts = {"relu": nn.ReLU, "leaky": lambda: nn.LeakyReLU(0.1),
            "gelu": nn.GELU, "silu": nn.SiLU, "elu": nn.ELU}

    torch.manual_seed(0)
    tl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0) # 0 cuz windows is so stupid
    vl = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)
    model = CNN(act=acts[act_name], drop=drop, width= width).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    EPOCHS = 20
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) # updates lr as the training happens
    crit = nn.CrossEntropyLoss()

    # training
    best_f1 = 0
    for epoch in range(EPOCHS):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
        sched.step()
        _, _, f1 = evaluate_with(model, vl)
        best_f1 = max(best_f1, f1)

        trial.report(f1, epoch)               # pruning
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_f1

study = optuna.create_study(direction="maximize",
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)) # objective of function is to max f1 score
study.optimize(objective, n_trials=40)
print(study.best_params, study.best_value)

[I 2026-10-21 17:46:14,850] A new study created in memory with name: no-name-640db69d-a3a6-44c3-894e-32e9e55e702c
[I 2026-10-21 17:53:21,501] Trial 0 finished with value: 0.9564206484094895 and parameters: {'lr': 0.0024089698686803745, 'wd': 0.0013989918572174349, 'drop': 0.008746813391433295, 'bs': 128, 'act': 'leaky', 'width': 16}. Best is trial 0 with value: 0.9564206484094895.
[I 2026-10-21 18:13:04,750] Trial 1 finished with value: 0.966476432885457 and parameters: {'lr': 0.005481312102487146, 'wd': 0.00021283568082662758, 'drop': 0.4962385289114958, 'bs': 64, 'act': 'silu', 'width': 48}. Best is trial 1 with value: 0.966476432885457.
[I 2026-10-21 18:27:14,321] Trial 2 finished with value: 0.9223123979322068 and parameters: {'lr': 0.00015928261718581765, 'wd': 1.8256598778837988e-06, 'drop': 0.3248269672324608, 'bs': 32, 'act': 'gelu', 'width': 16}. Best is trial 1 with value: 0.966476432885457.
[I 2026-10-21 18:36:47,483] Trial 3 finished with value: 0.9480652056118285 and param

{'lr': 0.0005982242953922088, 'wd': 0.0006351779679560851, 'drop': 0.3164256663260222, 'bs': 32, 'act': 'leaky', 'width': 48} 0.9753966790470778


params: {'lr': 0.0005982242953922088, 'wd': 0.0006351779679560851, 'drop': 0.3164256663260222, 'bs': 32, 'act': 'leaky', 'width': 48} best f1-score: 0.9753966790470778

In [36]:
# same thing but with accuracy as the metric
def objective(trial):
    # params to optimize for
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True) # learning rate
    wd = trial.suggest_float("wd", 1e-6, 1e-2, log=True) # weight decay
    drop = trial.suggest_float("drop", 0.0, 0.5) # dropout vals
    bs = trial.suggest_categorical("bs", [32, 64, 128]) # batch size
    act_name = trial.suggest_categorical("act", ["relu", "leaky", "gelu", "silu", "elu"]) # activation
    width = trial.suggest_categorical("width", [16, 32, 48])
    acts = {"relu": nn.ReLU, "leaky": lambda: nn.LeakyReLU(0.1),
            "gelu": nn.GELU, "silu": nn.SiLU, "elu": nn.ELU}

    torch.manual_seed(0)
    tl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0) # 0 cuz windows is so stupid
    vl = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)
    model = CNN(act=acts[act_name], drop=drop, width=width).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    EPOCHS = 20
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) # updates lr as the training happens
    crit = nn.CrossEntropyLoss()

    # training
    best_acc = 0
    for epoch in range(EPOCHS):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
        sched.step()
        val_loss, acc, f1 = evaluate_with(model, vl)
        best_acc = max(best_acc, acc)

        trial.report(acc, epoch)               # pruning on accuracy
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acc

study = optuna.create_study(direction="maximize",
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)) # maximize val accuracy

# start from f1 best params so this search can only match or beat them
study.enqueue_trial({"lr": 0.0005982242953922088, "wd": 0.0006351779679560851, "drop": 0.3164256663260222, "bs": 32, "act": "leaky", "width": 48})

study.optimize(objective, n_trials=40)
print(study.best_params, study.best_value)

[I 2026-10-22 03:00:05,906] A new study created in memory with name: no-name-975f8da9-6590-46c2-8fe6-ff4708fb26d2
[I 2026-10-22 03:13:40,694] Trial 0 finished with value: 0.9731838127742565 and parameters: {'lr': 0.0005982242953922088, 'wd': 0.0006351779679560851, 'drop': 0.3164256663260222, 'bs': 32, 'act': 'leaky', 'width': 48}. Best is trial 0 with value: 0.9731838127742565.
[I 2026-10-22 03:20:14,007] Trial 1 finished with value: 0.9726962457337884 and parameters: {'lr': 0.002259557281175199, 'wd': 1.043607631782549e-06, 'drop': 0.28870567936045355, 'bs': 64, 'act': 'silu', 'width': 32}. Best is trial 0 with value: 0.9731838127742565.
[I 2026-10-22 03:26:36,047] Trial 2 finished with value: 0.9512432959531936 and parameters: {'lr': 0.0002933627486279358, 'wd': 0.0008004729884530769, 'drop': 0.10784790339902106, 'bs': 128, 'act': 'leaky', 'width': 32}. Best is trial 0 with value: 0.9731838127742565.
[I 2026-10-22 03:36:16,990] Trial 3 finished with value: 0.9726962457337884 and para

{'lr': 0.0007696547987689935, 'wd': 1.2741295663408004e-05, 'drop': 0.3314296892372199, 'bs': 32, 'act': 'leaky', 'width': 48} 0.977571916138469


accuracy params: {'lr': 0.0007696547987689935, 'wd': 1.2741295663408004e-05, 'drop': 0.3314296892372199, 'bs': 32, 'act': 'leaky', 'width': 48}
accuracy : 0.977571916138469

params are actually very similar to the f1 params except weight decay which is significantly smaller

In [38]:
# actual train loop now that we have our hyperparams
# f1 score model:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128,     shuffle=False, num_workers=0)

cnn_model = CNN(act=lambda: nn.LeakyReLU(0.1), drop=0.3164256663260222, width=48).to(device)
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=0.0005982242953922088,
                              weight_decay=0.0006351779679560851)

num_epochs = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
patience = 15
best_val_loss = float('inf')
epochs_no_improve = 0
f1_model_state = None

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    # train
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss_val = loss(outputs, labels)
        loss_val.backward()
        optimizer.step()

        running_loss += loss_val.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    cnn_model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)

            val_running_loss += loss(outputs, labels).item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        f1_model_state = copy.deepcopy(cnn_model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

cnn_model.load_state_dict(f1_model_state)
torch.save(f1_model_state, "f1_model.pt")

Epoch 1/50 | train_loss: 0.5487 train_acc: 0.8037 | val_loss: 0.5600 val_acc: 0.8006
Epoch 2/50 | train_loss: 0.3438 train_acc: 0.8765 | val_loss: 0.3396 val_acc: 0.8835
Epoch 3/50 | train_loss: 0.2814 train_acc: 0.8991 | val_loss: 0.4206 val_acc: 0.8537
Epoch 4/50 | train_loss: 0.2395 train_acc: 0.9146 | val_loss: 0.1976 val_acc: 0.9283
Epoch 5/50 | train_loss: 0.2166 train_acc: 0.9254 | val_loss: 0.2656 val_acc: 0.9049
Epoch 6/50 | train_loss: 0.2023 train_acc: 0.9273 | val_loss: 0.1874 val_acc: 0.9371
Epoch 7/50 | train_loss: 0.1919 train_acc: 0.9342 | val_loss: 0.2408 val_acc: 0.9118
Epoch 8/50 | train_loss: 0.1846 train_acc: 0.9374 | val_loss: 0.1513 val_acc: 0.9483
Epoch 9/50 | train_loss: 0.1696 train_acc: 0.9362 | val_loss: 0.2610 val_acc: 0.9083
Epoch 10/50 | train_loss: 0.1703 train_acc: 0.9411 | val_loss: 0.1880 val_acc: 0.9313
Epoch 11/50 | train_loss: 0.1557 train_acc: 0.9472 | val_loss: 0.1340 val_acc: 0.9527
Epoch 12/50 | train_loss: 0.1488 train_acc: 0.9477 | val_loss: 

In [39]:
# accuracy model
# exact same code just different params
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128,     shuffle=False, num_workers=0)

cnn_model = CNN(act=lambda: nn.LeakyReLU(0.1), drop= 0.3314296892372199, width=48).to(device)
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=0.0007696547987689935,
                              weight_decay=1.2741295663408004e-05)

num_epochs = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
patience = 15
best_val_loss = float('inf')
epochs_no_improve = 0
acc_model_state = None

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    # train
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss_val = loss(outputs, labels)
        loss_val.backward()
        optimizer.step()

        running_loss += loss_val.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    cnn_model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)

            val_running_loss += loss(outputs, labels).item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        acc_model_state = copy.deepcopy(cnn_model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

cnn_model.load_state_dict(acc_model_state)
torch.save(acc_model_state, "accuracy_model.pt")

Epoch 1/50 | train_loss: 0.5573 train_acc: 0.8030 | val_loss: 0.7986 val_acc: 0.7284
Epoch 2/50 | train_loss: 0.3550 train_acc: 0.8742 | val_loss: 0.3118 val_acc: 0.8874
Epoch 3/50 | train_loss: 0.2858 train_acc: 0.8993 | val_loss: 0.4482 val_acc: 0.8464
Epoch 4/50 | train_loss: 0.2492 train_acc: 0.9085 | val_loss: 0.1910 val_acc: 0.9352
Epoch 5/50 | train_loss: 0.2317 train_acc: 0.9184 | val_loss: 0.2041 val_acc: 0.9210
Epoch 6/50 | train_loss: 0.1993 train_acc: 0.9282 | val_loss: 0.3163 val_acc: 0.8961
Epoch 7/50 | train_loss: 0.1979 train_acc: 0.9302 | val_loss: 0.1448 val_acc: 0.9522
Epoch 8/50 | train_loss: 0.1859 train_acc: 0.9324 | val_loss: 0.2559 val_acc: 0.9098
Epoch 9/50 | train_loss: 0.1766 train_acc: 0.9380 | val_loss: 0.1542 val_acc: 0.9405
Epoch 10/50 | train_loss: 0.1636 train_acc: 0.9419 | val_loss: 0.1578 val_acc: 0.9430
Epoch 11/50 | train_loss: 0.1543 train_acc: 0.9450 | val_loss: 0.1726 val_acc: 0.9425
Epoch 12/50 | train_loss: 0.1467 train_acc: 0.9481 | val_loss: 

In [40]:
# test set predictions

test_files = sorted(p.name for p in test_folder.glob("*.png"))
test_df = pd.DataFrame({image_id_col: test_files}) # sorted by the id of the image
test_loader = DataLoader(CellDataset(test_df, test_folder, eval_tf),
                         batch_size=128, shuffle=False, num_workers=0)

# f1 model results
cnn_model.load_state_dict(torch.load("f1_model.pt"))
cnn_model.eval()

probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.softmax(cnn_model(x.to(device)), dim=1).cpu()) # softmax here in test since we no longer have cross entropy with built in softmax
probs = torch.cat(probs).numpy() 
preds = probs.argmax(1) # pick highest probability class

f1_results = pd.DataFrame({image_id_col: test_files, "label": preds})
f1_results.to_csv("f1_submission.csv", index=False)

In [41]:
# accuracy results
cnn_model.load_state_dict(torch.load("accuracy_model.pt"))
cnn_model.eval()

probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.softmax(cnn_model(x.to(device)), dim=1).cpu()) # softmax here in test since we no longer have cross entropy with built in softmax
probs = torch.cat(probs).numpy() 
preds = probs.argmax(1) # pick highest probability class

acc_results = pd.DataFrame({image_id_col: test_files, "label": preds})
acc_results.to_csv("accuracy_submission.csv", index=False)

using tta to improve accuracy (takes multiple pics and uses the average during training), using accuracy model (since it yielded higher scores)

In [45]:
def tta_probs(model, loader):
    model.eval(); out = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device); p = 0
            for k in range(4):                                   # 0, 90, 180, 270 degrees
                r = torch.rot90(x, k, dims=(2, 3))
                p = p + torch.softmax(model(r), 1) + torch.softmax(model(torch.flip(r, dims=(3,))), 1)
            out.append((p / 8).cpu())                            # average of 8 views
    return torch.cat(out)

cnn_model = CNN(act=lambda: nn.LeakyReLU(0.1), drop=0.3314296892372199, width=48).to(device)
cnn_model.load_state_dict(torch.load("accuracy_model.pt"))
cnn_model.eval()

test_files = sorted(p.name for p in test_folder.glob("*.png"))
test_df = pd.DataFrame({image_id_col: test_files})
test_loader = DataLoader(CellDataset(test_df, test_folder, eval_tf),
                         batch_size=128, shuffle=False, num_workers=0)

# tta predictions
probs = tta_probs(cnn_model, test_loader).numpy()
preds = probs.argmax(1)

tta_results = pd.DataFrame({image_id_col: test_files, "label": preds})

tta_results.to_csv("tta_accuracy_submission.csv", index=False)

KeyboardInterrupt: 

In [48]:
# accuracy model
# exact same code just different params but more epochs
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128,     shuffle=False, num_workers=0)

cnn_model = CNN(act=lambda: nn.LeakyReLU(0.1), drop= 0.3314296892372199, width=48).to(device)
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=0.0007696547987689935,
                              weight_decay=1.2741295663408004e-05)

num_epochs = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
patience = 15
best_val_loss = float('inf')
epochs_no_improve = 0
acc_model_state = None

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    # train
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss_val = loss(outputs, labels)
        loss_val.backward()
        optimizer.step()

        running_loss += loss_val.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    cnn_model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)

            val_running_loss += loss(outputs, labels).item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        acc_model_state = copy.deepcopy(cnn_model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

cnn_model.load_state_dict(acc_model_state)
torch.save(acc_model_state, "accuracy_model_100epochs.pt")

Epoch 1/100 | train_loss: 0.5418 train_acc: 0.8092 | val_loss: 0.4423 val_acc: 0.8372
Epoch 2/100 | train_loss: 0.3466 train_acc: 0.8769 | val_loss: 0.3829 val_acc: 0.8581
Epoch 3/100 | train_loss: 0.2795 train_acc: 0.8986 | val_loss: 0.2520 val_acc: 0.9176
Epoch 4/100 | train_loss: 0.2475 train_acc: 0.9124 | val_loss: 0.2179 val_acc: 0.9254
Epoch 5/100 | train_loss: 0.2301 train_acc: 0.9188 | val_loss: 0.1939 val_acc: 0.9391
Epoch 6/100 | train_loss: 0.2042 train_acc: 0.9266 | val_loss: 0.2253 val_acc: 0.9215
Epoch 7/100 | train_loss: 0.1942 train_acc: 0.9303 | val_loss: 0.2189 val_acc: 0.9254
Epoch 8/100 | train_loss: 0.1849 train_acc: 0.9350 | val_loss: 0.1438 val_acc: 0.9527
Epoch 9/100 | train_loss: 0.1865 train_acc: 0.9332 | val_loss: 0.1379 val_acc: 0.9537
Epoch 10/100 | train_loss: 0.1724 train_acc: 0.9393 | val_loss: 0.1704 val_acc: 0.9371
Epoch 11/100 | train_loss: 0.1625 train_acc: 0.9411 | val_loss: 0.1660 val_acc: 0.9376
Epoch 12/100 | train_loss: 0.1554 train_acc: 0.9449 

In [49]:
# accuracy results
cnn_model.load_state_dict(torch.load("accuracy_model_100epochs.pt"))
cnn_model.eval()

probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.softmax(cnn_model(x.to(device)), dim=1).cpu()) # softmax here in test since we no longer have cross entropy with built in softmax
probs = torch.cat(probs).numpy() 
preds = probs.argmax(1) # pick highest probability class

acc_results = pd.DataFrame({image_id_col: test_files, "label": preds})
acc_results.to_csv("accuracy_100epochs_submission.csv", index=False)

In [50]:
# since no overfitting was observed, we'll try a final model with all the train data (no val)
full_ds = CellDataset(df, train_folder, train_tf)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=True, num_workers=0)

cnn_model = CNN(act=lambda: nn.LeakyReLU(0.1), drop= 0.3314296892372199, width=48).to(device)
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=0.0007696547987689935,
                              weight_decay=1.2741295663408004e-05)

num_epochs = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
patience = 15
best_val_loss = float('inf')
epochs_no_improve = 0
acc_model_state = None

train_losses = []
train_accs = []

for epoch in range(num_epochs):
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in full_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss_val = loss(outputs, labels)
        loss_val.backward()
        optimizer.step()

        running_loss += loss_val.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    acc_model_state = copy.deepcopy(cnn_model.state_dict())

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f}")

cnn_model.load_state_dict(acc_model_state)
torch.save(acc_model_state, "accuracy_model_full_dataset.pt")

Epoch 1/100 | train_loss: 0.5039 train_acc: 0.8183
Epoch 2/100 | train_loss: 0.3224 train_acc: 0.8870
Epoch 3/100 | train_loss: 0.2673 train_acc: 0.9018
Epoch 4/100 | train_loss: 0.2312 train_acc: 0.9190
Epoch 5/100 | train_loss: 0.2118 train_acc: 0.9233
Epoch 6/100 | train_loss: 0.1988 train_acc: 0.9304
Epoch 7/100 | train_loss: 0.2001 train_acc: 0.9291
Epoch 8/100 | train_loss: 0.1772 train_acc: 0.9381
Epoch 9/100 | train_loss: 0.1667 train_acc: 0.9418
Epoch 10/100 | train_loss: 0.1546 train_acc: 0.9474
Epoch 11/100 | train_loss: 0.1507 train_acc: 0.9469
Epoch 12/100 | train_loss: 0.1508 train_acc: 0.9480
Epoch 13/100 | train_loss: 0.1452 train_acc: 0.9498
Epoch 14/100 | train_loss: 0.1375 train_acc: 0.9525
Epoch 15/100 | train_loss: 0.1341 train_acc: 0.9539
Epoch 16/100 | train_loss: 0.1287 train_acc: 0.9561
Epoch 17/100 | train_loss: 0.1270 train_acc: 0.9567
Epoch 18/100 | train_loss: 0.1237 train_acc: 0.9569
Epoch 19/100 | train_loss: 0.1170 train_acc: 0.9600
Epoch 20/100 | train_

In [51]:
# accuracy results
cnn_model.load_state_dict(torch.load("accuracy_model_full_dataset.pt"))
cnn_model.eval()

probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.softmax(cnn_model(x.to(device)), dim=1).cpu()) # softmax here in test since we no longer have cross entropy with built in softmax
probs = torch.cat(probs).numpy() 
preds = probs.argmax(1) # pick highest probability class

acc_results = pd.DataFrame({image_id_col: test_files, "label": preds})
acc_results.to_csv("accuracy_fulldataset_submission.csv", index=False)